# Baseline Model

In this notebook, we will train a traditional machine learning model before using transformers.

## Step 1: Load the dataset.

In [3]:
from datasets import load_dataset

dataset = load_dataset("dair-ai/emotion")
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 16000
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 2000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 2000
    })
})

## Step 2: Inspect examples.

In [4]:
dataset["train"][0]

{'text': 'i didnt feel humiliated', 'label': 0}

In [5]:
dataset["train"][10]

{'text': 'i feel like i have to make the suffering i m seeing mean something',
 'label': 0}

## Step 3: Get the label names.

In [6]:
label_names = dataset["train"].features["label"].names
label_names

['sadness', 'joy', 'love', 'anger', 'fear', 'surprise']

In [7]:
label_names[0]

'sadness'

## Step 4: Print text with emotion name

In [8]:
example = dataset["train"][0]

text = example["text"]
label = example["label"]
emotion = label_names[label]

print(f'Text: "{text}"')
print("Label number:", label)
print("Emotion:", emotion)

Text: "i didnt feel humiliated"
Label number: 0
Emotion: sadness


## Step 5: Create X and y

In [9]:
x_train = dataset["train"]["text"]
y_train = dataset["train"]["label"]

x_test = dataset["test"]["text"]
y_test = dataset["test"]["label"]

print("First training text:", x_train[0])
print("First training label:", y_train[0])
print("First training emotion:", label_names[y_train[0]])

print("Number of training examples:", len(x_train))
print("Number of test examples:", len(x_test))

First training text: i didnt feel humiliated
First training label: 0
First training emotion: sadness
Number of training examples: 16000
Number of test examples: 2000


## Step 6: Understand TF-IDF

In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer

tFidf = TfidfVectorizer()

x_train_tFidf = tFidf.fit_transform(x_train)

x_train_tFidf.shape

(16000, 15186)

## Step 7: Inspect the vocabulary

In [11]:
feature_names = tFidf.get_feature_names_out()

feature_names[:20]

array(['aa', 'aaaaaaand', 'aaaaand', 'aaaand', 'aac', 'aahhh', 'aaron',
       'ab', 'abandon', 'abandoned', 'abandoning', 'abandonment',
       'abated', 'abbigail', 'abc', 'abdomen', 'abdominal', 'abducted',
       'abelard', 'abhorrent'], dtype=object)

In [12]:
feature_names[-20:]

array(['zimbabwe', 'zimbabweans', 'zimmerman', 'zine', 'zipline', 'zoll',
       'zombie', 'zombies', 'zone', 'zoned', 'zonisamide', 'zoo', 'zoom',
       'zooming', 'zooms', 'zq', 'zucchini', 'zum', 'zumba', 'zz'],
      dtype=object)

## Step 8: Train Logic Regression

Now that the text has been converted into numbers with TF-IDF, we can train a traditional machine learning model.

Logistic Regression will learn which words are connected with each emotion.

In [13]:
from sklearn.linear_model import LogisticRegression

baseline_model = LogisticRegression(max_iter=1000)

baseline_model.fit(x_train_tFidf, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`mul

## Step 9: Make Predictions On Test Data

The model was trained with TF-IDF numbes, so the test text must also be converted into TF-IDF numbers before prediction.

In [14]:
x_test_tFidf = tFidf.transform(x_test)

y_pred = baseline_model.predict(x_test_tFidf)

y_pred[:10]

array([0, 0, 0, 1, 0, 4, 3, 1, 1, 1])

## Step 10: Compare Predictions With True Labels

Now we compare what the model predicted with the correct labeks frim the test dataset.

In [15]:
for i in range(10):
    text = x_test[i]
    true_label = y_test[i]
    predicted_label = y_pred[i]

    print(f'Text: "{text}"')
    print("True emotion:", label_names[true_label])
    print("Predicted emotion:", label_names[predicted_label])
    print()

Text: "im feeling rather rotten so im not very ambitious right now"
True emotion: sadness
Predicted emotion: sadness

Text: "im updating my blog because i feel shitty"
True emotion: sadness
Predicted emotion: sadness

Text: "i never make her separate from me because i don t ever want her to feel like i m ashamed with her"
True emotion: sadness
Predicted emotion: sadness

Text: "i left with my bouquet of red and yellow tulips under my arm feeling slightly more optimistic than when i arrived"
True emotion: joy
Predicted emotion: joy

Text: "i was feeling a little vain when i did this one"
True emotion: sadness
Predicted emotion: sadness

Text: "i cant walk into a shop anywhere where i do not feel uncomfortable"
True emotion: fear
Predicted emotion: fear

Text: "i felt anger when at the end of a telephone call"
True emotion: anger
Predicted emotion: anger

Text: "i explain why i clung to a relationship with a boy who was in many ways immature and uncommitted despite the excitement i shoul

### Reflection: True Labels and Model Mistakes

The true emotion comes from the dataset label. In this project, "true emotion" means the emotion that was already assigned by the dataset.

This does not mean the label is always perfect. Emotion can be subjective, and some texts may contain mixed emotions, humor, sarcasm, or missing context.

The model is learning patterns from the training data. It is not learning perfect human emotion. It is learning how to imitate the labels in this dataset.

When the predicted emotion is different from the true emotion, it can mean the model made a mistake, but it can also mean the text is ambiguous or difficult to classify.

## Step 11: Evaluate The Model

Now we calculate metrics to understand how well the baseline model perfoms on the test data.

Accurary shows the percentage of predictions that were correct.

The Classification report gives more detailed metrics for each emotion:

- precision: when the model predicts this emotion, how often is it correct?
- recall: out of all real examples of this emotion, how many did the model find?
- f1-score: balance between precision and recall
- support: how many test examples exist for this emotion

In [16]:
from sklearn.metrics import accuracy_score, classification_report

accuaracy = accuracy_score(y_test, y_pred)

print("Accracy:", accuaracy)
print()
print(classification_report(y_test, y_pred, target_names=label_names))

Accracy: 0.8615

              precision    recall  f1-score   support

     sadness       0.89      0.93      0.91       581
         joy       0.83      0.96      0.89       695
        love       0.82      0.58      0.68       159
       anger       0.90      0.81      0.85       275
        fear       0.87      0.78      0.82       224
    surprise       0.84      0.47      0.60        66

    accuracy                           0.86      2000
   macro avg       0.86      0.75      0.79      2000
weighted avg       0.86      0.86      0.86      2000



### Result

The baseline model reached an accuacy of 0.8615, meaning it correctl predicted about 86% of the test examples.

The model performs best on emotions with more examples, such as sadness and joy.

The model performs worse on emotions with fewer examples, especially surprise and love. This shows that class imbalance can affect model performance.

Accuracy alone is not enought to understand the model. The classifications report is important because it shows performance for each emotion separately.

## Step 12: Inspect Wrong Predictions

Now we create a table with the test texts, the true emotions, and the predicted emotions.

Then we filter only the examples where the model prediction was different from the true label.

This helps us understand what kinds of sentences are difficult from the baseline model.

In [22]:
import pandas as pd

results_df = pd.DataFrame({
    "text": x_test,
    "true_label": [label_names[label] for label in y_test],
    "predicted_label": [label_names[label] for label in y_pred]
})

mistakes_df = results_df[results_df["true_label"] != results_df["predicted_label"]]

mistakes_df.head(10)

,text,true_label,predicted_label
9,i jest i feel grumpy tired and pre menstrual w...,anger,joy
10,i don t feel particularly agitated,fear,anger
12,i pay attention it deepens into a feeling of b...,fear,sadness
20,im not sure the feeling of loss will ever go a...,sadness,joy
40,i feel if i completely hated things i d exerci...,anger,sadness
49,i was feeling pretty anxious all day but my fi...,fear,joy
59,i think about the things ive said and the stuf...,anger,joy
62,i spent wandering around still kinda dazed and...,joy,sadness
65,i feel a little stunned but can t imagine what...,surprise,sadness
67,i feel a bit stressed even though all the thin...,anger,sadness


### Result

The wrong predictions show that the baseline model has difficulty with ambiguous or mixed-emotion texts.

Some emotions share similar language. For example, fear, anger, and sadness can all include negative words, so the model may confuse them.

The model also struggles with context. Since TF-IDF focuses on word patterns, it may give too much importance to individual words and miss the full meaning of the sentence.

Surprise is also difficult because there are fewer examples of this class in the dataset.

This mistake analysis is important because it shows why a transformer model may perform better latter: transformers can use more context from the full sentence.

## Baseline Concepts Learned

### TF-IDF numbers

Machine learning models do not understand text directly, so we need to convert words into numbers.

TF-IDF does this by giving importance to words based on how useful they are in the dataset. Common words usually get less importance, while more specific words can help the model identify emotions.

### Why use notebooks?

Notebooks are useful because they let us write code, see outputs, create chats, and write explanations in the same place.

In this project, the notebook helps me learn the proccess step by step: loading data, undertanding labels, training the model, evaluating results, and writing observations.

### fit_transform vs transform

`fit_transform` learns from the data and converts it into numbers.

`transform` only converts data using what was already learned.

That is why we use `fit_transform` on the training texts, because TF-IDF needs to learn vocabulary. For the test texts, we use only `transform`, so the model uses the same vocabulary from training.

### class imbalance

Class imbalance happens when some classes have may more examples tha others.
In this dataset, emotions like sadness and joy have more examples than surprise. This can make the model perform better on common emotions and worse on rare emotions.